In [56]:
import os
import openai
from dotenv import load_dotenv


In [57]:
load_dotenv()
openai.api_key = os.getenv("OPENAI_API_KEY")

In [58]:
import http.client
import json
import requests
from datetime import datetime


def get_coordinates(city):

    url = "https://nominatim.openstreetmap.org/search"

    params = {
        "q": city,
        "format": "json",
        "limit": 1
    }

    headers = {
        "User-Agent": "weather-app"
    }

    response = requests.get(
        url,
        params=params,
        headers=headers
    )

    data = response.json()

    if not data:
        raise Exception("City not found")

    lat = data[0]["lat"]
    lon = data[0]["lon"]

    return lat, lon


def get_current_weather(city):

    lat, lon = get_coordinates(city)
    date = datetime.now().strftime("%Y-%m-%d")

    conn = http.client.HTTPSConnection(
        "ai-weather-by-meteosource.p.rapidapi.com"
    )

    headers = {
        "x-rapidapi-key": "f0f710fafcmsh8ef5cbcc9bbecd7p18f21djsn676755497097",
        "x-rapidapi-host": "ai-weather-by-meteosource.p.rapidapi.com",
        "Content-Type": "application/json"
    }

    conn.request(
       "GET", f"/time_machine?lat={lat}&lon={lon}&date={date}&units=auto", headers=headers
    )

    res = conn.getresponse()
    data = res.read()

    return json.loads(data.decode("utf-8"))

In [63]:
functions = [
    {
        "name": "get_current_weather",
        "description": "Get the current weather in a given location",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string",
                    "description": "The city to get the weather for"
                }
            },
            "required": ["city"]
        }
    }
]

In [92]:
user_message = "what is the weather in LUCKNOW?"
messages = []
messages.append({"role": "user", "content": user_message})
completion = openai.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages,
    functions=functions,
    function_call="auto")

In [86]:
completion

ChatCompletion(id='chatcmpl-ENNTDraITwr7RtG6G7XwidQbwh41T', choices=[Choice(finish_reason='function_call', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=FunctionCall(arguments='{"city":"Etawah"}', name='get_current_weather'), tool_calls=None))], created=1789240471, model='gpt-4o-mini-2024-07-18', object='chat.completion', metadata=None, moderation=None, service_tier='default', system_fingerprint='fp_8c012bb801', usage=CompletionUsage(completion_tokens=16, prompt_tokens=64, total_tokens=80, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0, text_tokens=None), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=None, cached_tokens=0, image_tokens=None, text_tokens=None)))

In [87]:
 response = completion.choices[0].message

In [93]:
import json

response = completion.choices[0].message
args = json.loads(
    response.to_dict()["function_call"]["arguments"]
)
city = args["city"]

In [94]:
function_result = get_current_weather(city)
messages.append(response)
messages.append(
    {
        "role": "function",
        "name": response.to_dict()["function_call"]["name"],
        "content": json.dumps(function_result)
    })

In [95]:
final_response = openai.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages,
    functions=functions,
)

In [82]:
print(final_response)

ChatCompletion(id='chatcmpl-ENNFztCiUvXQkmPbLrl7KUHVKxCRO', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='The current weather in Delhi is as follows:\n\n- **Temperature**: 28.5°C\n- **Feels Like**: 34.5°C\n- **Weather**: Partly sunny\n- **Humidity**: 82%\n- **Wind**: 2.1 m/s from the East-Southeast\n- **Cloud Cover**: 25%\n- **Pressure**: 1009 hPa\n- **Precipitation**: None\n\nTonight, the temperature will remain warm, dropping to around 28.0°C later in the evening, with partly sunny conditions expected.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1789239651, model='gpt-4o-mini-2024-07-18', object='chat.completion', metadata=None, moderation=None, service_tier='default', system_fingerprint='fp_b70318dca3', usage=CompletionUsage(completion_tokens=124, prompt_tokens=3865, total_tokens=3989, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0,

In [83]:
print(final_response.choices[0].message)

ChatCompletionMessage(content='The current weather in Delhi is as follows:\n\n- **Temperature**: 28.5°C\n- **Feels Like**: 34.5°C\n- **Weather**: Partly sunny\n- **Humidity**: 82%\n- **Wind**: 2.1 m/s from the East-Southeast\n- **Cloud Cover**: 25%\n- **Pressure**: 1009 hPa\n- **Precipitation**: None\n\nTonight, the temperature will remain warm, dropping to around 28.0°C later in the evening, with partly sunny conditions expected.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None)


In [96]:
msg = final_response.choices[0].message
content = getattr(msg, "content", None) or (msg.get("content") if isinstance(msg, dict) else None)

if content:
    print(content)
elif getattr(msg, "function_call", None):
    fc = msg.function_call
    print("Model requested function call:", getattr(fc, "name", None))
    print("Arguments:", getattr(fc, "arguments", None))
else:
    print(msg)

The current weather in Lucknow is as follows:

- **Condition:** Cloudy
- **Temperature:** 27.4°C
- **Feels Like:** 31.8°C
- **Humidity:** 88%
- **Wind Speed:** 2.7 m/s (from the East-Northeast)
- **Cloud Cover:** 85%
- **Pressure:** 1009 hPa

The weather is expected to remain predominantly cloudy throughout the day with some variations in temperature.
